In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import sys
import time

# Analysis of the results of the CSV'S resulting from the sim

In this notebook it will be shown the process of analyzing the resulting sim data and stablishing a pipeline which will lead to a .py projectwhich will automatically run after the GAMA sim and be able to extract the results via graphics and similar

First there is a lot of csvs. we need to joint them all into one

In [41]:
BASE_SIM_DIR = "20260329_182320"
#HARDCODDED NUMBER OF SIMS
N_SIMS = 1000

columns = ["simulation_id", "bus_code", "travels_completed","stop_num", "direction", "current_time","timeslot","boards","residual_boards","dropoffs", "occupation", "time_between_stops"]

joint_dataframe_results = pd.DataFrame(columns=columns)

dataframes_to_join_results = []
for iteration in range(0,N_SIMS):
    new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/RESULTS/log_Simulation {iteration}.csv", skiprows = 1,
            names = columns, encoding="utf-8")
    dataframes_to_join_results.append(new_dataframe)



joint_dataframe_results = pd.concat(data for data in dataframes_to_join_results)


In [42]:
N_SIMS = 1000


columns = ["simulation_id", "bus_code","travels_completed", "stop_num", "direction","timeslot", "event_code", "time_event","param_event_1", "param_event_2"]

joint_dataframe_events = pd.DataFrame(columns=columns)

dataframes_to_join_events = []
for iteration in range(0,N_SIMS):
    new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/EVENTS/log_Simulation {iteration}.csv",skiprows = 1,
                         names = columns, encoding="utf-8")
    dataframes_to_join_events.append(new_dataframe)



joint_dataframe_events = pd.concat(data for data in dataframes_to_join_events)

In [43]:
columns = ["simulation_id", "bus_code","travels_completed", "stop_num", "timeslot", "current_time","list_passengers_time"]

joint_dataframe_passengers = pd.DataFrame(columns=columns)

dataframes_to_join_passengers = []
for iteration in range(0,N_SIMS):
    new_dataframe = pd.read_csv(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PASSENGERS/log_Simulation {iteration}.csv", skiprows = 1,
                names=columns, encoding="utf-8")
    dataframes_to_join_passengers.append(new_dataframe)



joint_dataframe_passengers = pd.concat(data for data in dataframes_to_join_passengers)


Once joined the data, its time to start making the graphics results andif well done, elevate the process to a .py code that accepts the ISODATE of the sim as a parameter, and build a directory which holds the graphics results.

## Graphic 1. Statistics of simulation Occupation VS REAL TIME occupation

In [44]:
# la idea de esto es generar gráficas a mansalva. La idea detrás ed esto es poder tener tantas gráficas como se guste y en la interfaz de usuario poder agrupar esas gráficas para que se den varios puntos de vista.


#idea 1. A parte de hacer gráficas para todos los timeslots, agrupar por hardcodded timestamps (Mañana[5], Hora punta[6,9], Valle de Mañana[10,13], Hora tarde[14,19], noche[20,23]
timeslots = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]



# Estadísticas generales. Medidas de Rendimiento. % de ocupación medio, general, por franja horaria, Enfrentamiento real vs teórico "normal" de ocupacion

In [45]:
#KPI 1. % de ocupación medio. % de ocupacion por franja horaria. % de ocupación de cada autobús en acda una de sus franjas
MAX_CAPACITY = 80

def time_to_seconds(time_str, include_seconds: bool):
    if include_seconds:
        h, m, s = map(int, time_str.split(":"))
    else:
        h,m = map(int, time_str.split(":"))
        s = 0
    return h * 3600 + m * 60 + s


#La easy.
percentage_occupation_per_stop = 0

#In toA

json_output_general_info = {}



data_toA = joint_dataframe_results[joint_dataframe_results["direction"] == "toA"]
data_toB = joint_dataframe_results[joint_dataframe_results["direction"] == "toB"]

json_output_general_info["Occupation_toA"] = (data_toA["occupation"].mean()/MAX_CAPACITY)*100
json_output_general_info["Occupation_toB"] = (data_toB["occupation"].mean()/MAX_CAPACITY)*100
json_output_general_info["Occupation"] = (joint_dataframe_results["occupation"].mean()/MAX_CAPACITY)*100



#Empiezan las graficassss
#graficas de evolucion a lo largo del tiempon de valor de %Occupation
list_occupation_values_toA = []
list_occupation_values_toB = []
list_occupation = []
for timeslot in timeslots:
    data_toA_per_ts = data_toA[data_toA["timeslot"] == timeslot]
    data_toB_per_ts = data_toB[data_toB["timeslot"] == timeslot]
    data_general = joint_dataframe_results[joint_dataframe_results["timeslot"] == timeslot]
    list_occupation_values_toA.append((data_toA_per_ts["occupation"].mean()/MAX_CAPACITY)*100)
    list_occupation_values_toB.append((data_toB_per_ts["occupation"].mean()/MAX_CAPACITY)*100)
    list_occupation.append((data_general["occupation"].mean()/MAX_CAPACITY)*100)


plt.style.use('seaborn-v0_8-whitegrid')

def save_barcode(horizontal_data, vertical_data, horizontal_label, vertical_label, leyend, label, filename):
    x = np.arange(len(horizontal_data))
    plt.figure(figsize=(12,5))
    plt.bar(x, vertical_data, width=0.4, label=leyend)
    plt.xticks(x + 0.2, horizontal_data, rotation=90)
    plt.ylabel(vertical_label)
    plt.xlabel(horizontal_label)
    plt.title(label)
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


save_barcode(timeslots, list_occupation_values_toA , "TimeSlots", "% of Occupation",  "Direction to A", "Bus occupation per timeslot", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/occupation_per_timeslot_toA.png")

save_barcode(timeslots, list_occupation_values_toB , "TimeSlots", "% of Occupation",  "Direction to B", "Bus occupation per timeslot", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/occupation_per_timeslot_toB.png")

save_barcode(timeslots, list_occupation , "TimeSlots", "% of Occupation",  "General Data", "Bus occupation per timeslot",
             f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/general/occupation_per_timeslot_general.png")


stops_list_toA = data_toA["stop_num"].unique()
stops_list_toB = data_toB["stop_num"].unique()

stops_list_toA = [int(data) for data in stops_list_toA]
stops_list_toB = [int(data) for data in stops_list_toB]


for stopcode in stops_list_toA:
    #obtain the data from that stop and extract per timeslots its value
    list_per_ts_per_stop = []
    data_stopcode = data_toA[data_toA["stop_num"] == stopcode]
    if stopcode == stops_list_toA[-1]:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/stops/occupation_per_timeslot_{stopcode}.png")
    else:

        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["occupation"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_A/stops/occupation_per_timeslot_{stopcode}.png")



for stopcode in stops_list_toB:
    #obtain the data from that stop and extract per timeslots its value
    list_per_ts_per_stop = []
    data_stopcode = data_toB[data_toB["stop_num"] == stopcode]
    if stopcode == stops_list_toB[-1]:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["dropoffs"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/stops/occupation_per_timeslot_{stopcode}.png")
    else:
        for timeslot in timeslots:
            data_per_ts_per_stop = data_stopcode[data_stopcode["timeslot"] == timeslot]
            list_per_ts_per_stop.append((data_per_ts_per_stop["occupation"].mean()/MAX_CAPACITY)*100)

        save_barcode(timeslots, list_per_ts_per_stop , "TimeSlots", "% of Occupation",  f"Stop {stopcode}", "Bus occupation per timeslot per stop", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/to_B/stops/occupation_per_timeslot_{stopcode}.png")



#Now occupation per physical bus
unique_bus = joint_dataframe_results["bus_code"].unique()


input_json_route = f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/includes/INPUTS/input_data_buses.json"

with open(input_json_route, "r") as f:
    input_json = json.load(f)

new_bus_json_data = {}


for bus in unique_bus:

    trip_occupation_percentage = 0

    data_per_bus = joint_dataframe_results[joint_dataframe_results["bus_code"] == bus]
    events_per_bus = joint_dataframe_events[joint_dataframe_events["bus_code"] == bus]

    unique_trips = data_per_bus['travels_completed'].unique()
    unique_trips = [int(data) for data in unique_trips]

    new_bus_json_data[f"{bus}"] = {}

    #Loop over each trip
    for trip in unique_trips:

        events_per_unique_trip =events_per_bus[events_per_bus["travels_completed"] == trip]

        total_percentage_of_delays_per_trip = (len(events_per_unique_trip)/N_SIMS)*100
        mean_delay_in_minutes_per_trip = (pd.Series([time_to_seconds(data, True) for data in events_per_unique_trip["time_event"]]) - pd.Series([time_to_seconds(data, False) for data in events_per_unique_trip["param_event_1"]])).mean()



        data_inputs = input_json[str(bus)][trip]

        data_simulation_per_trip = data_per_bus[data_per_bus['travels_completed'] == trip]
        trip_occupation_percentage = (data_simulation_per_trip["occupation"].mean()/MAX_CAPACITY)*100

        events_per_trip_per_bus = events_per_bus[events_per_bus['travels_completed'] == trip]

        mean_time_of_trip = 0
        #Extract the mean ocupation for saving the plot

        list_mean_values = []
        direction = ""
        if int(data_inputs[0]) == stops_list_toA[0]:
            direction = "to_A"

            #extract the mean estimated time. Calculating the mean arrival time to stop stops_list_toA[-1] and the mean departure time from stops_list_toA[-1]
            data_per_trip_last_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toA[-1]]
            time_per_last_stop = data_per_trip_last_stop["current_time"]
            time_per_last_stop = [time_to_seconds(data, True) for data in time_per_last_stop]
            data_per_trip_first_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toA[0]]
            time_per_first_stop = data_per_trip_first_stop["current_time"]
            time_per_first_stop = [time_to_seconds(data, True) for data in time_per_first_stop]
            mean_time_of_trip = sum(time_per_last_stop)/len(time_per_last_stop) - sum(time_per_first_stop)/len(time_per_first_stop)

            for stop in stops_list_toA:
                data_per_trip_per_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stop]
                list_mean_values.append( (data_per_trip_per_stop["occupation"].mean()/MAX_CAPACITY)*100)

            save_barcode(stops_list_toA, list_mean_values , "Stops in toA Direction", "% of Occupation",  f"Occupation for trip", f"Bus occupation per stop for unique bus {bus} with estimate departure {data_inputs[1]}", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/mean_occupation_per_stop_toA_{bus}_{trip}.png")




        else:
            direction = "toB"

            #extract the mean estimated time. Calculating the mean arrival time to stop stops_list_toA[-1] and the mean departure time from stops_list_toA[-1]
            data_per_trip_last_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toB[-1]]
            time_per_last_stop = data_per_trip_last_stop["current_time"]
            time_per_last_stop = [time_to_seconds(data, True) for data in time_per_last_stop]
            data_per_trip_first_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stops_list_toB[0]]
            time_per_first_stop = data_per_trip_first_stop["current_time"]
            time_per_first_stop = [time_to_seconds(data, True) for data in time_per_first_stop]
            mean_time_of_trip = sum(time_per_last_stop)/len(time_per_last_stop) - sum(time_per_first_stop)/len(time_per_first_stop)


            for stop in stops_list_toB:
                data_per_trip_per_stop = data_simulation_per_trip[data_simulation_per_trip["stop_num"] == stop]
                list_mean_values.append( (data_per_trip_per_stop["occupation"].mean()/MAX_CAPACITY)*100)

            save_barcode(stops_list_toB, list_mean_values , "Stops in toB Direction", "% of Occupation",  f"Occupation for trip", f"Bus occupation per stop for unique bus {bus} with estimate departure {data_inputs[1]}", f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/mean_occupation_per_stop_toB_{bus}_{trip}.png")

        #print(f"for trip {data_inputs[0]}-{data_inputs[1]} for bus {bus} we have a mean time of trip {mean_time_of_trip}")
        #Now data_inputs will act as the key for the data for that specific trip. The data will be. Mean % of occupation. (Mean % of occupation per stop(indicating direction)) Mean %of delays and Mean minutes of delay in case it exists.
        new_bus_json_data[f"{bus}"][f"{data_inputs[0]}-{data_inputs[1]}"] = {"Mean_Occupation_Percentage": trip_occupation_percentage, "Direction": direction, "Mean_time_of_trip": mean_time_of_trip, "Percentage_sims_delay_departure": total_percentage_of_delays_per_trip, "Mean_Minutes_departure_delay" : mean_delay_in_minutes_per_trip}


#Important KPI. Theoretical vs Real Distribution of Occupation per Timeslot per Stop


#Time between stops.

FGAGGGGGGGGGG Series([], Name: time_event, dtype: object)
FGAGGGGGGGGGG 0    7:02:05
0    7:03:35
0    7:03:25
0    7:00:55
0    7:00:45
      ...   
0    7:01:05
0    7:01:15
0    6:58:25
0    6:58:05
0    6:58:10
Name: time_event, Length: 992, dtype: object
FGAGGGGGGGGGG 1    7:32:25
1    7:35:15
1    7:33:45
1    7:32:10
1    7:32:20
      ...   
1    7:31:50
1    7:32:00
1    7:33:25
1    7:33:05
1    7:36:10
Name: time_event, Length: 162, dtype: object
FGAGGGGGGGGGG Series([], Name: time_event, dtype: object)
FGAGGGGGGGGGG Series([], Name: time_event, dtype: object)
FGAGGGGGGGGGG 1    7:49:15
1    7:53:15
1    7:51:05
3    7:49:55
1    7:50:45
      ...   
1    7:51:20
1    7:47:10
1    7:53:35
2    7:51:15
1    7:47:40
Name: time_event, Length: 798, dtype: object
FGAGGGGGGGGGG 2    8:22:10
2    8:26:10
2    8:24:10
4    8:25:20
2    8:30:15
      ...   
2    8:22:35
2    8:25:40
2    8:21:25
2    8:26:25
3    8:23:35
Name: time_event, Length: 616, dtype: object
FGAGGGGGGGGGG Seri

In [47]:
print(new_bus_json_data)

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/buses/general_info.json", "w") as f:
    json.dump(new_bus_json_data, f)

with open(f"{os.getcwd()}/logs/SIMULATION_{BASE_SIM_DIR}/PLOTS/json_output_general_info.json", "w") as f:
    json.dump(json_output_general_info, f)

{'3569': {'2646-6:25': {'Mean_Occupation_Percentage': np.float64(13.751916666666666), 'Direction': 'to_A', 'Mean_time_of_trip': 2133.665000000001, 'Percentage_sims_delay_departure': 0.0, 'Mean_Minutes_departure_delay': nan}, '5727-6:55': {'Mean_Occupation_Percentage': np.float64(5.8390625), 'Direction': 'toB', 'Mean_time_of_trip': 1627.635000000002, 'Percentage_sims_delay_departure': 99.2, 'Mean_Minutes_departure_delay': np.float64(347.4899193548387)}, '2646-7:30': {'Mean_Occupation_Percentage': np.float64(11.147083333333335), 'Direction': 'to_A', 'Mean_time_of_trip': 2632.4550000000017, 'Percentage_sims_delay_departure': 16.2, 'Mean_Minutes_departure_delay': np.float64(177.6851851851852)}, '5727-8:30': {'Mean_Occupation_Percentage': np.float64(3.614791666666667), 'Direction': 'toB', 'Mean_time_of_trip': 1733.9800000000032, 'Percentage_sims_delay_departure': 0.0, 'Mean_Minutes_departure_delay': nan}}, '3584': {'2646-7:00': {'Mean_Occupation_Percentage': np.float64(77.29275), 'Direction